# Paper-ready hallucination detector evaluation

This notebook runs a **full, reproducible evaluation** suitable for a report or paper:

- **Data**: All four datasets (HaluEval, MedHallu, Med-HALT, MedHal) — download, preprocess, validate.
- **Features**: Extract hidden states from all datasets with a larger sample size (2000 per dataset) using `facebook/opt-125m`.
- **Transfer matrix**: Train a probe on each dataset and evaluate on **all** datasets (N×N).
- **Multiple seeds**: Run with 3 seeds and report **mean ± std** for AUROC and F1.
- **Ablations**: Feature and layer ablations.
- **Note**: Med-HALT's 100% positive (hallucination) subset is documented; use for out-of-domain eval only or balance as needed.

**Setup**:
1. Mount Google Drive and set `PROJECT_DIR` below to your `hallucination-detector` folder.
2. Replace `"xxx_here_bitte"` with your HuggingFace token in the code cell (or add a Colab secret named `HF_TOKEN`).
3. Run the single code cell below.

**Debug (Colab troubleshooting):** Set `DEBUG = True` for extra diagnostics (paths, env, feature files). Set `SHOW_TRACEBACK = True` to print full tracebacks on errors.

In [ ]:
# ========== DEBUG FLAGS (set True if something goes wrong on Colab) ==========
DEBUG = False           # True: print paths, env, feature files, config
SHOW_TRACEBACK = False   # True: print full traceback on errors

def _debug_print(*args, **kwargs):
    if DEBUG:
        print("[DEBUG]", *args, **kwargs)

def _run(cmd, description="run"):
    _debug_print(f"Running: {cmd}")
    r = subprocess.run(cmd, shell=True, cwd=PROJECT_DIR)
    if r.returncode != 0:
        print(f"[ERROR] {description} failed with exit code {r.returncode}")
    _debug_print(f"Exit code: {r.returncode}")
    return r

# ========== 1) SETUP ==========
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import subprocess
import traceback
import yaml
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/hallucination-detector"  # set to your project path
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# HuggingFace token: replace "xxx_here_bitte" with your token, or use Colab secret "HF_TOKEN"
HF_TOKEN = "xxx_here_bitte"
try:
    from google.colab import userdata
    os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    _debug_print("Colab userdata not used:", e)
    if HF_TOKEN and HF_TOKEN != "xxx_here_bitte":
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    elif os.environ.get("HF_TOKEN"):
        os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

!pip install -q -r requirements.txt
!pip install -q bitsandbytes
print("cwd:", os.getcwd())

# ---------- Diagnostics (when DEBUG=True) ----------
if DEBUG:
    print("\n--- Diagnostics ---")
    print("Python:", sys.version)
    print("PROJECT_DIR:", PROJECT_DIR)
    print("PROJECT_DIR exists:", Path(PROJECT_DIR).exists())
    print("HF token set:", bool(os.environ.get("HUGGING_FACE_HUB_TOKEN") or os.environ.get("HF_TOKEN")))
    for d in ["configs", "scripts", "src", "data/processed", "data/features", "results"]:
        p = Path(PROJECT_DIR) / d
        print(f"  {d}: exists={p.exists()}" + (f" (files: {len(list(p.iterdir()))})" if p.exists() and p.is_dir() else ""))
    print("------------------\n")

# ========== 2) DATA: DOWNLOAD + PREPROCESS ==========
try:
    r = _run("python scripts/01_download_datasets.py", "01_download_datasets")
    if r.returncode != 0 and not DEBUG:
        print("Tip: Set DEBUG=True to see more. Check HF token and network.")
    if r.returncode != 0:
        raise RuntimeError(f"01_download_datasets failed with exit code {r.returncode}")
    r = _run("python scripts/02_preprocess.py --out data/processed", "02_preprocess")
    if r.returncode != 0:
        raise RuntimeError(f"02_preprocess failed with exit code {r.returncode}")
    if DEBUG:
        for f in Path(PROJECT_DIR).glob("data/processed/*.csv"):
            print(f"  {f}: {f.stat().st_size} bytes")
except Exception as e:
    print("[ERROR] Data download/preprocess failed:", e)
    if SHOW_TRACEBACK:
        traceback.print_exc()
    raise

# ========== 3) FEATURE EXTRACTION (all datasets, larger sample) ==========
MAX_SAMPLES = 2000
MODEL_SHORT = "opt-125m"
try:
    r = subprocess.run(
        f"python scripts/03_extract_features.py --model facebook/opt-125m --datasets halueval medhallu medhalt medhal --max_samples {MAX_SAMPLES} --batch_size 8",
        shell=True,
        cwd=PROJECT_DIR,
    )
    if r.returncode != 0:
        print("[ERROR] Feature extraction failed. Check GPU/memory and that datasets loaded (01/02).")
        if SHOW_TRACEBACK:
            traceback.print_exc()
        raise RuntimeError(f"03_extract_features failed with exit code {r.returncode}")
    if DEBUG:
        features_dir_early = Path(PROJECT_DIR) / "data" / "features"
        if features_dir_early.exists():
            for f in sorted(features_dir_early.iterdir()):
                print(f"  {f.name}: {f.stat().st_size} bytes")
        else:
            print("  data/features/ does not exist yet")
except Exception as e:
    print("[ERROR] Feature extraction failed:", e)
    if SHOW_TRACEBACK:
        traceback.print_exc()
    raise

# ========== 4) FULL TRANSFER MATRIX WITH MULTIPLE SEEDS ==========
try:
    config_path = Path(PROJECT_DIR) / "configs/config.yaml"
    if not config_path.exists():
        raise FileNotFoundError(f"Config not found: {config_path}")
    with open(config_path) as f:
        config = yaml.safe_load(f)
    features_dir = Path(PROJECT_DIR) / config.get("output", {}).get("features_dir", "data/features")
    results_dir = Path(PROJECT_DIR) / config.get("output", {}).get("results_dir", "results")
    results_dir.mkdir(parents=True, exist_ok=True)
    if DEBUG:
        print("config.model:", config.get("model", {}).get("name"))
        print("config layers:", config.get("model", {}).get("layers_to_extract"))
        print("features_dir:", features_dir, "exists:", features_dir.exists())
        print("results_dir:", results_dir)

    all_datasets = [
        d for d in ["halueval", "medhallu", "medhalt", "medhal"]
        if (features_dir / f"{d}_labels.npy").exists()
    ]
    print("Datasets with features:", all_datasets)
    if not all_datasets:
        print(f"[WARN] No feature files found. Expect *_labels.npy and *_{MODEL_SHORT}_layer*_mean.npy under", features_dir)
        if DEBUG:
            print("Listing features_dir:", list(features_dir.iterdir()) if features_dir.exists() else "dir missing")
except Exception as e:
    print("[ERROR] Config/paths failed:", e)
    if SHOW_TRACEBACK:
        traceback.print_exc()
    raise

seeds = [42, 43, 44]
try:
    from src.evaluation.transfer import run_transfer_experiment
except ImportError as e:
    print("[ERROR] Could not import run_transfer_experiment. Is cwd/PROJECT_DIR correct?", e)
    if SHOW_TRACEBACK:
        traceback.print_exc()
    raise

all_dfs = []
try:
    for seed in seeds:
        for train_ds in all_datasets:
            _debug_print(f"Transfer: seed={seed} train={train_ds}")
            df = run_transfer_experiment(
                train_ds,
                all_datasets,
                config,
                features_dir,
                model_short=MODEL_SHORT,
                random_state=seed,
            )
            if not df.empty:
                df["seed"] = seed
                all_dfs.append(df)
except Exception as e:
    print("[ERROR] Transfer experiment failed:", e)
    if SHOW_TRACEBACK:
        traceback.print_exc()
    raise

if all_dfs:
    big = pd.concat(all_dfs, ignore_index=True)
    summary = (
        big.groupby(["train_dataset", "eval_dataset"])
        .agg(
            auroc_mean=("auroc", "mean"),
            auroc_std=("auroc", "std"),
            f1_mean=("f1", "mean"),
            f1_std=("f1", "std"),
            n_train=("n_train", "first"),
            n_eval=("n_eval", "first"),
        )
        .reset_index()
    )
    summary["auroc"] = summary.apply(
        lambda r: f"{r['auroc_mean']:.3f} ± {r['auroc_std']:.3f}" if pd.notna(r["auroc_std"]) else f"{r['auroc_mean']:.3f}",
        axis=1,
    )
    summary["f1"] = summary.apply(
        lambda r: f"{r['f1_mean']:.3f} ± {r['f1_std']:.3f}" if pd.notna(r["f1_std"]) else f"{r['f1_mean']:.3f}",
        axis=1,
    )
    print("\n=== Transfer matrix (mean ± std over seeds) ===")
    print(summary[["train_dataset", "eval_dataset", "auroc", "f1", "n_train", "n_eval"]].to_string(index=False))
    summary.to_csv(results_dir / "transfer_matrix_seeded.csv", index=False)
    big.to_csv(results_dir / "transfer_matrix_raw.csv", index=False)
    print(f"\nSaved: {results_dir}/transfer_matrix_seeded.csv, transfer_matrix_raw.csv")
else:
    print("No transfer results (missing feature files?). Run 03 with all datasets.")

# ========== 5) ABLATIONS ==========
if all_datasets:
    try:
        r = subprocess.run(
            f"python scripts/06_ablations.py --model_short {MODEL_SHORT} --train_dataset halueval --eval_datasets {' '.join(all_datasets)}",
            shell=True,
            cwd=PROJECT_DIR,
        )
        if r.returncode != 0:
            print("[WARN] Ablations script exited with code", r.returncode)
        elif DEBUG:
            print("Ablation files:", list(Path(results_dir).glob("ablation*.csv")))
    except Exception as e:
        print("[ERROR] Ablations failed:", e)
        if SHOW_TRACEBACK:
            traceback.print_exc()

# ========== 6) SUMMARY ==========
print("\n=== Paper-ready evaluation complete ===")
print("• Transfer matrix: train on each dataset, evaluate on all; reported as mean ± std over 3 seeds.")
print("• Med-HALT: subset used has 100% positive (hallucination) labels; interpret as out-of-domain eval or balance for training.")
print("• Ablations: results/ablation_features.csv, results/ablation_layers.csv")
if DEBUG:
    print("\n--- Final debug: results dir contents ---")
    for f in sorted(Path(results_dir).iterdir()):
        print(f"  {f.name}: {f.stat().st_size} bytes")
